In [1]:
import duckdb
from pathlib import Path
con = duckdb.connect('../data/cfpb.duckdb')
con.sql("SELECT COUNT(*) AS rows FROM complaints_scoped").show()

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 533015 │
└────────┘



In [4]:
def run_sql_file(name):
    text = Path(f'../sql/{name}').read_text()
    for stmt in text.split(';'):
        lines = stmt.splitlines()
        body = '\n'.join(l for l in lines if not l.strip().startswith('--')).strip()
        if not body:
            continue
        labels = [l for l in lines if l.strip().startswith('--')]
        print(labels[-1] if labels else '')
        rel = con.sql(stmt)
        if rel is not None:
            display(rel.df())

In [3]:
con.sql("SELECT company_response, COUNT(*) AS n FROM complaints_scoped GROUP BY 1 ORDER BY 2 DESC").show()
con.sql("SELECT timely_response, COUNT(*) AS n FROM complaints_scoped GROUP BY 1 ORDER BY 2 DESC").show()

┌─────────────────────────────────┬────────┐
│        company_response         │   n    │
│             varchar             │ int64  │
├─────────────────────────────────┼────────┤
│ Closed with explanation         │ 426748 │
│ Closed with monetary relief     │  69924 │
│ Closed with non-monetary relief │  35393 │
│ Untimely response               │    815 │
│ In progress                     │    135 │
└─────────────────────────────────┴────────┘

┌─────────────────┬────────┐
│ timely_response │   n    │
│     varchar     │ int64  │
├─────────────────┼────────┤
│ Yes             │ 527053 │
│ No              │   5962 │
└─────────────────┴────────┘



In [5]:
run_sql_file('04_trend.sql')

-- 0. Analysis view: full months only (Aug 2026 is a partial month)


,Count


-- 1. Monthly volume by product with MoM change and 3-month rolling average


,month,product,complaints,complaints_ex_surge,mom_change,mom_pct,rolling_3m
0,2023-09-01,Checking or savings account,3771,3771,<NA>,NaN,3771.0
1,2023-10-01,Checking or savings account,3766,3766,-5,-0.1,3769.0
2,2023-11-01,Checking or savings account,3744,3744,-22,-0.6,3760.0
3,2023-12-01,Checking or savings account,3792,3792,48,1.3,3767.0
4,2024-01-01,Checking or savings account,3993,3993,201,5.3,3843.0
...,...,...,...,...,...,...,...
135,2026-03-01,Prepaid card,460,460,-36,-7.3,533.0
136,2026-04-01,Prepaid card,455,455,-5,-1.1,470.0
137,2026-05-01,Prepaid card,499,499,44,9.7,471.0
138,2026-06-01,Prepaid card,528,528,29,5.8,494.0


In [6]:
run_sql_file('05_top_issues.sql')


-- 1. Top 10 issues per product, ranked by volume


,product,issue,complaints,pct_of_product,rnk
0,Checking or savings account,Managing an account,109814,56.8,1
1,Checking or savings account,Closing an account,26751,13.8,2
2,Checking or savings account,Problem with a lender or other company chargin...,25595,13.2,3
3,Checking or savings account,Opening an account,17052,8.8,4
4,Checking or savings account,Problem caused by your funds being low,14211,7.3,5
5,Credit card,Problem with a purchase shown on your statement,57650,33.1,1
6,Credit card,"Other features, terms, or problems",23750,13.6,2
7,Credit card,Getting a credit card,23623,13.6,3
8,Credit card,Fees or interest,21571,12.4,4
9,Credit card,Closing your account,15169,8.7,5


-- 2. Running share: how quickly the top issues add up to the whole product


,product,issue_rank,issue,complaints,cum_pct
0,Checking or savings account,1,Managing an account,109814,56.8
1,Checking or savings account,2,Closing an account,26751,70.6
2,Checking or savings account,3,Problem with a lender or other company chargin...,25595,83.8
3,Checking or savings account,4,Opening an account,17052,92.7
4,Checking or savings account,5,Problem caused by your funds being low,14211,100.0
5,Credit card,1,Problem with a purchase shown on your statement,57650,33.1
6,Credit card,2,"Other features, terms, or problems",23750,46.7
7,Credit card,3,Getting a credit card,23623,60.2
8,Credit card,4,Fees or interest,21571,72.6
9,Credit card,5,Closing your account,15169,81.3


In [8]:
run_sql_file('06_growth.sql')

-- 1. Fastest growing issues by percent


,product,issue,prior_12m,last_12m,change,growth_pct
0,"Money transfer, virtual currency, or money ser...","Managing, opening, or closing your mobile wall...",1258,2305,1047,83.2
1,"Money transfer, virtual currency, or money ser...",Trouble accessing funds in your mobile or digi...,2199,3927,1728,78.6
2,Credit card,Problem with a purchase shown on your statement,16467,28368,11901,72.3
3,Checking or savings account,Problem with a lender or other company chargin...,7818,12193,4375,56.0
4,"Money transfer, virtual currency, or money ser...",Unauthorized transactions or other transaction...,4279,6330,2051,47.9
5,Credit card,"Advertising and marketing, including promotion...",2703,3903,1200,44.4
6,"Money transfer, virtual currency, or money ser...",Lost or stolen money order,205,290,85,41.5
7,Checking or savings account,Managing an account,35210,49107,13897,39.5
8,Prepaid card,Problem with a purchase or transfer,1574,2184,610,38.8
9,"Money transfer, virtual currency, or money ser...",Money was not available when promised,2047,2740,693,33.9


-- 2. Biggest absolute increases


,product,issue,prior_12m,last_12m,change,growth_pct
0,Checking or savings account,Managing an account,35210,49107,13897,39.5
1,Credit card,Problem with a purchase shown on your statement,16467,28368,11901,72.3
2,Checking or savings account,Problem with a lender or other company chargin...,7818,12193,4375,56.0
3,Checking or savings account,Closing an account,8831,11449,2618,29.6
4,Credit card,"Other features, terms, or problems",7601,10141,2540,33.4
5,"Money transfer, virtual currency, or money ser...",Unauthorized transactions or other transaction...,4279,6330,2051,47.9
6,Credit card,Fees or interest,7383,9171,1788,24.2
7,"Money transfer, virtual currency, or money ser...",Trouble accessing funds in your mobile or digi...,2199,3927,1728,78.6
8,"Money transfer, virtual currency, or money ser...",Fraud or scam,9771,11179,1408,14.4
9,Checking or savings account,Opening an account,5905,7164,1259,21.3


In [9]:
run_sql_file('07_peer_compare.sql')

-- 1. Issue mix gap, all fintech companies


,topic,fintech_pct,bank_pct,gap_pts
0,"Money transfer, virtual currency, or money ser...",15.7,1.8,13.9
1,Credit card | Problem with a purchase shown on...,3.0,15.8,-12.8
2,"Money transfer, virtual currency, or money ser...",10.4,0.5,9.9
3,"Money transfer, virtual currency, or money ser...",12.8,3.9,8.9
4,Checking or savings account | Managing an account,19.2,26.7,-7.5
5,"Money transfer, virtual currency, or money ser...",6.8,0.1,6.8
6,"Credit card | Other features, terms, or problems",0.6,6.7,-6.1
7,Credit card | Getting a credit card,0.7,6.0,-5.3
8,Credit card | Fees or interest,0.2,4.5,-4.3
9,"Money transfer, virtual currency, or money ser...",4.1,0.0,4.1


-- 2. Same comparison without Block (Block is over half of all fintech complaints)


,topic,fintech_pct,bank_pct,gap_pts
0,Credit card | Problem with a purchase shown on...,4.1,15.8,-11.7
1,"Money transfer, virtual currency, or money ser...",8.2,0.5,7.7
2,"Money transfer, virtual currency, or money ser...",7.6,0.1,7.5
3,"Money transfer, virtual currency, or money ser...",11.3,3.9,7.4
4,"Money transfer, virtual currency, or money ser...",8.0,1.8,6.2
5,"Credit card | Other features, terms, or problems",0.8,6.7,-6.0
6,Credit card | Getting a credit card,0.9,6.0,-5.2
7,Credit card | Fees or interest,0.3,4.5,-4.2
8,"Money transfer, virtual currency, or money ser...",4.0,0.0,3.9
9,Credit card | Closing your account,0.4,4.0,-3.6


In [10]:
run_sql_file('08_outcomes.sql')

-- 0. Resolved complaints with outcome flags


,Count


-- 1. Outcomes by product


,product,complaints,relief_pct,untimely_pct,poor_outcome_pct
0,Checking or savings account,193396,19.3,0.7,80.8
1,Credit card,174309,29.7,0.8,70.5
2,"Money transfer, virtual currency, or money ser...",93008,10.2,2.2,89.9
3,Prepaid card,18494,26.5,5.7,73.5


-- 2. Worst five issues per product by poor outcome rate (issues with 500 or more complaints)


,product,issue,complaints,relief_pct,untimely_pct,poor_outcome_pct,poor_rank
0,Checking or savings account,Opening an account,17047,16.6,0.5,83.5,1
1,Checking or savings account,Problem with a lender or other company chargin...,25589,18.8,0.8,81.3,2
2,Checking or savings account,Managing an account,109804,19.4,0.7,80.7,3
3,Checking or savings account,Closing an account,26747,20.4,0.9,79.7,4
4,Checking or savings account,Problem caused by your funds being low,14209,20.6,0.4,79.5,5
5,Credit card,Trouble using your card,8099,15.7,0.9,84.4,1
6,Credit card,Struggling to pay your bill,3036,18.1,0.7,81.9,2
7,Credit card,Closing your account,15169,20.1,0.8,79.9,3
8,Credit card,"Advertising and marketing, including promotion...",8650,26.5,0.8,73.6,4
9,Credit card,"Other features, terms, or problems",23748,27.6,0.8,72.4,5


-- 3. Fintech vs Bank


,company_group,complaints,relief_pct,untimely_pct,poor_outcome_pct
0,Fintech,91698,10.7,1.6,89.3
1,Bank,180409,27.7,0.2,72.4


-- 4. Each peer company with 1000 or more complaints


,company_group,company,complaints,relief_pct,untimely_pct,poor_outcome_pct
0,Bank,JPMORGAN CHASE & CO.,44703,15.8,0.0,84.2
1,Bank,WELLS FARGO & COMPANY,33976,18.9,0.0,81.1
2,Bank,CAPITAL ONE FINANCIAL CORPORATION,33796,20.8,0.0,79.2
3,Bank,"BANK OF AMERICA, NATIONAL ASSOCIATION",34754,41.0,0.8,59.5
4,Bank,"CITIBANK, N.A.",33180,46.0,0.0,54.0
5,Fintech,ROBINHOOD MARKETS INC.,3866,0.0,9.7,100.0
6,Fintech,"Block, Inc.",37583,0.1,2.4,99.9
7,Fintech,TransferWise Ltd,1367,9.0,7.0,92.2
8,Fintech,"SOFI TECHNOLOGIES, INC.",5332,8.3,0.1,91.7
9,Fintech,Chime Financial Inc,19228,9.2,0.0,90.8


In [11]:
run_sql_file('09_priority.sql')

-- 1. Metrics per topic


,Count
0,28


-- 2. Fixability ratings from 0 to 1, left empty for now so every topic defaults to 0.5


,Count


-- 3. Score and rank


,Count
0,28


-- 4. Top 25 candidates


,priority_rank,product,issue,complaints,growth_pct,poor_outcome_pct,fixability,score
0,1,Checking or savings account,Managing an account,109804,39.4,80.7,0.5,0.696
1,2,Checking or savings account,Problem with a lender or other company chargin...,25589,55.9,81.3,0.5,0.689
2,3,"Money transfer, virtual currency, or money ser...",Fraud or scam,26147,14.4,91.1,0.5,0.672
3,4,"Money transfer, virtual currency, or money ser...",Unauthorized transactions or other transaction...,12007,47.9,85.6,0.5,0.654
4,5,Credit card,Problem with a purchase shown on your statement,57647,72.3,66.4,0.5,0.639
5,6,"Money transfer, virtual currency, or money ser...","Managing, opening, or closing your mobile wall...",4258,83.1,87.6,0.5,0.631
6,7,Checking or savings account,Closing an account,26747,29.6,79.7,0.5,0.609
7,8,"Money transfer, virtual currency, or money ser...",Other transaction problem,25369,-52.9,93.7,0.5,0.594
8,9,"Money transfer, virtual currency, or money ser...",Trouble accessing funds in your mobile or digi...,7285,78.5,83.4,0.5,0.591
9,10,Credit card,Trouble using your card,8099,33.5,84.4,0.5,0.556
